In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Definimos el modelo Transformer

In [2]:
class TransformerModel(nn.Module):
  def __init__(self,input_dim,hidden_dim,num_heads,num_layers,output_dim):
      super(TransformerModel, self).__init__()

      # Capa de embedding
      self.embedding = nn.Embedding(input_dim, hidden_dim)

      # Capa de Transformer (con Encoder y Decoder)
      self.transformer = nn.Transformer(d_model=hidden_dim,
                                            nhead=num_heads,
                                            num_encoder_layers=num_layers,
                                            num_decoder_layers=num_layers)

      # Capa de salida (para la predicción final)
      self.fc_out = nn.Linear(hidden_dim, output_dim)

  def forward(self, src, tgt):
        # Embeddings para la secuencia de entrada y la secuencia de salida
        src = self.embedding(src)  # (seq_len, batch, hidden_dim)
        tgt = self.embedding(tgt)  # (seq_len, batch, hidden_dim)

        # Transformer genera una representación codificada
        output = self.transformer(src, tgt)  # (seq_len, batch, hidden_dim)

        # Tomamos la última salida del decoder para la predicción
        out = self.fc_out(output[-1, :, :])  # (batch, output_dim)
        return out

In [3]:
# Datos de ejemplo
input_dim = 5  # Se puede pensar como el vocabulario de números (1, 2, 3, 4, 5)
output_dim = 1  # Solo predecimos un valor
hidden_dim = 16  # Dimensión interna de los embeddings
num_heads = 2  # Número de cabezas de atención
num_layers = 2  # Número de capas en el transformer

# Crear el modelo Transformer
model = TransformerModel(input_dim=input_dim,
                         hidden_dim=hidden_dim,
                         num_heads=num_heads,
                         num_layers=num_layers,
                         output_dim=output_dim)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [4]:
# Optimización y criterio
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# Datos de entrada: secuencia y objetivo (por ejemplo, [1, 2, 3] → 4)
src = torch.tensor([[1, 2, 3]]).T  # (3, 1) -> Forma correcta para el transformer
tgt = torch.tensor([[2, 3, 4]]).T  # (3, 1) -> Forma correcta para el transformer


In [6]:
# Entrenamiento
num_epochs = 300
for epoch in range(num_epochs):
    model.train()

    # Forward pass
    output = model(src, tgt)

    # Cálculo de la pérdida
    loss = criterion(output, torch.tensor([[5.0]]))  # Queremos predecir 5

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Predicción: {output.item():.4f}")

Epoch 0, Loss: 23.7222, Predicción: 0.1295
Epoch 50, Loss: 4.0618, Predicción: 2.9846
Epoch 100, Loss: 0.9358, Predicción: 4.0326
Epoch 150, Loss: 0.2106, Predicción: 4.5411
Epoch 200, Loss: 0.0028, Predicción: 4.9470
Epoch 250, Loss: 0.0114, Predicción: 4.8934
